In [0]:
%pip install yfinance

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS oil_stock")
spark.sql("USE CATALOG oil_stock")

In [0]:
import yfinance as yf
import pandas as pd

frames = []

tickers = ["EQNR.OL", "BZ=F"]

for ticker in tickers:
    df = yf.download(
        ticker,
        period="30d",
        interval="5m",
        auto_adjust=False
    )

    df = df.reset_index()

    # Remove yfinance's extra column level
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["symbol"] = ticker

    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

display(raw)

In [0]:
from pyspark.sql import functions as F

bronze_df = (
    spark.createDataFrame(raw)
    .select(
        F.col("Datetime").alias("datetime"),
        F.col("symbol"),
        F.col("Open").alias("open"),
        F.col("High").alias("high"),
        F.col("Low").alias("low"),
        F.col("Close").alias("close"),
        F.col("Volume").alias("volume")
    )
    .withColumn(
        "currency",
        F.when(F.col("symbol") == "EQNR.OL", "NOK")
         .when(F.col("symbol") == "BZ=F", "USD")
    )
    .withColumn("source", F.lit("yfinance"))
    .withColumn("ingested_at", F.current_timestamp())
)
display(bronze_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

bronze_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "bronze.market_5m"
)